In [9]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 1) True nonlinear system (pendulum)
# ============================================================
def plant_dynamics(x, u, b=0.1, m=0.200, g=9.81, l=0.5):
    """Continuous dynamics: x = [theta, omega]. Returns x_dot."""
    x1, x2 = x
    x1_dot = x2
    x2_dot = -b/m * x2 - g/l * np.sin(x1)  # u=0 (free pendulum)
    return np.array([x1_dot, x2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for a 2-state system (no inputs):
    z = [S(x1), S(x2), S(x1)S(x2), S(x1)^2, S(x2)^2, 1]
    """
    s_x1 = sigmoidal(x_est[0])
    s_x2 = sigmoidal(x_est[1])
    return np.array([s_x1, s_x2, s_x1*s_x2, s_x1**2, s_x2**2, 1.0])

def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured output at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]

        z_i = construct_z_vector(x_state_for_z)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance
            P_pred = self.P[i] + self.Q[i]

            # Innovation covariance (scalar)
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-12:
                M_i = 1e-12

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update
            self.weights[i] += self.eta * K_i * e_i

            # Covariance update (Joseph or simple; we symmetrize to keep numerical hygiene)
            P_update = P_pred - np.outer(K_i, (H_i.T @ P_pred).ravel())
            self.P[i] = 0.5 * (P_update + P_update.T)  # enforce symmetry

# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]
        z = construct_z_vector(x_state_for_z)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.01

    # --- True system init ---
    x_true = np.zeros((n_steps, 2))
    x_true[0] = [np.pi / 4, 0.0]  # 45 degrees, zero velocity
    u = 0.0

    # --- RHONN config ---
    num_neurons = 2
    num_features = 6
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    # np.random.seed(12345)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # --- EKF ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=0.8
    )
    x_hat_ekf = np.zeros((n_steps, 2))
    x_hat_ekf[0] = x_true[0]

    # --- PF ---
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=300,
        initial_weights=common_initial_weights,
        Q_std=0.05, R_std=np.sqrt(2e-2), ess_threshold=150  # ESS < N/2
    )

    # Force identical particle initialization if desired:
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 2))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)

        # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])

        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured output at k
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])

        # ---- 3) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

    # ============================================================
    # 6) Results & plots
    # ============================================================
    mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
    mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
    mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
    mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)

    print(f"\nFinal EKF-RHONN Weights (Neuron 1 for x1): {ekf_trainer.weights[0]}")
    print(f"Final EKF-RHONN Weights (Neuron 2 for x2): {ekf_trainer.weights[1]}")
    print(f"Final PF-RHONN Weight Estimates (Neuron 1 for x1): {pf_trainer.get_estimate()[0]}")
    print(f"Final PF-RHONN Weight Estimates (Neuron 2 for x2): {pf_trainer.get_estimate()[1]}")

    print("\n--- Performance Comparison (MSE) ---")
    print(f"EKF MSE x1 (Angle):     {mse_x1_ekf:.6f}")
    print(f"EKF MSE x2 (Ang. Vel.): {mse_x2_ekf:.6f}")
    print(f"PF  MSE x1 (Angle):     {mse_x1_pf:.6f}")
    print(f"PF  MSE x2 (Ang. Vel.): {mse_x2_pf:.6f}")

    states_info = [
        {'idx': 0, 'var': 'θ', 'desc': 'Pendulum Angle', 'y_label': 'Angle (rad)',
         'chi': 'χ₁ (True Angle)', 'x': 'x₁ (Est. Angle)'},
        {'idx': 1, 'var': 'ω', 'desc': 'Pendulum Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)',
         'chi': 'χ₂ (True Ang. Vel.)', 'x': 'x₂ (Est. Ang. Vel.)'}
    ]

    for state_info in states_info:
        i = state_info['idx']
        trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                                 name=state_info['chi'], line=dict(color='black', width=2))
        trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                              name=f"{state_info['x']} (PF)", line=dict(dash='dot'))
        trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                               name=f"{state_info['x']} (EKF)", line=dict(dash='dash'))

        fig = go.Figure([trace_plant, trace_pf, trace_ekf])
        fig.update_layout(
            title=f'Plant vs RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
            xaxis_title='Time (s)',
            yaxis_title=state_info['y_label'],
            legend=dict(x=0, y=1, orientation='h'),
            font=dict(size=12),
            plot_bgcolor='white',
            paper_bgcolor='white'
        )
        fig.show()

    # Errors
    error_x1_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
    error_x2_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
    error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
    error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=t_history, y=error_x1_ekf, mode='lines',
                              name=f'EKF Error x1 (MSE={mse_x1_ekf:.6f})', opacity=0.7))
    fig2.add_trace(go.Scatter(x=t_history, y=error_x1_pf, mode='lines',
                              name=f'PF Error x1 (MSE={mse_x1_pf:.6f})', opacity=0.7))
    fig2.add_trace(go.Scatter(x=t_history, y=error_x2_ekf, mode='lines',
                              name=f'EKF Error x2 (MSE={mse_x2_ekf:.6f})', opacity=0.7))
    fig2.add_trace(go.Scatter(x=t_history, y=error_x2_pf, mode='lines',
                              name=f'PF Error x2 (MSE={mse_x2_pf:.6f})', opacity=0.7))
    fig2.update_layout(
        title='Identification Errors',
        xaxis_title='Time (s)',
        yaxis_title='Error',
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig2.show()


Common Initial Weights:
  Neuron 0: [ 0.29779099  0.02654836 -0.3119291   0.21441461 -0.42127451  0.2615406 ]
  Neuron 1: [-0.21509889 -0.08733517  0.36838267 -0.28112765 -0.30998091  0.47158924]
Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulation finished.

Final EKF-RHONN Weights (Neuron 1 for x1): [ 2.04843841e+00 -1.36873288e-01  1.84737674e-03  1.93874239e+00
  1.73641287e-01 -1.49248137e+00]
Final EKF-RHONN Weights (Neuron 2 for x2): [-3.20530209  4.78138809  0.16707566  2.17158486 -0.66647072 -1.17783163]
Final PF-RHONN Weight Estimates (Neuron 1 for x1): [ 0.11578474  0.09472665  0.96987862  2.81987503 -0.6493018  -0.87296159]
Final PF-RHONN Weight Estimates (Neuron 2 for x2): [-1.37731137  2.75118163  1.22127931 -1.12076116 -0.271